# Personal case — NPCRA on Samsung Galaxy Watch data

Full chronobiological analysis on personal actigraphy data ingested by `n24sal.io.samsung`. **This notebook is shipped without executed outputs** to avoid committing personal data into the repository — run it locally against your own `data/personal/<subject_id>/activity.parquet`.

**Prerequisites** :

```bash
python -m n24sal.io.samsung ingest /path/to/SamsungHealth/ \
    --subject-id S001 --output data/personal/S001/ \
    --age 38 --sex M --diagnosis N24SWD -vv
```

Once the parquet exists, run all cells. Optional: edit the `SUBJECT_ID` variable below if you use a different identifier.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

from n24sal.npcra import (
    bootstrap_tau_ci,
    circadian_function_index,
    estimate_tau,
    interdaily_stability,
    intradaily_variability,
    l5_m10,
    relative_amplitude,
)
from n24sal.viz import (
    apply_theme,
    average_24h_profile,
    coverage_heatmap,
    double_plot_actogram,
    m10_phase_drift_plot,
)
from n24sal.viz.theme import AMBER, CYAN, TEAL

apply_theme("dark")
pio.renderers.default = "notebook_connected"

SUBJECT_ID = "S001"
DATA_DIR = Path("..") / "data" / "personal" / SUBJECT_ID
EPOCHS_PER_HOUR = 60
EPOCHS_PER_DAY = 1440

if not (DATA_DIR / "activity.parquet").exists():
    raise FileNotFoundError(
        f"{DATA_DIR/'activity.parquet'} missing — run `python -m n24sal.io.samsung ingest ...` first"
    )

activity = pd.read_parquet(DATA_DIR / "activity.parquet")
metadata = json.loads((DATA_DIR / "subject_metadata.json").read_text())
coverage = json.loads((DATA_DIR / "coverage_report.json").read_text())
TIMEZONE = metadata.get("timezone", "Europe/Paris")
print(f"Subject {metadata['subject_id']}  |  device {metadata['device']}  |  tz {TIMEZONE}")
print(f"Recording: {metadata['recording_start']}  →  {metadata['recording_end']}")
print(f"Epochs: {len(activity):,} ({coverage['coverage_pct']:.1f}% coverage over {coverage['duration_days']} days)")
print(f"Gaps > 2 epochs: {coverage['n_gaps_over_2_epochs']} ; longest gap: {coverage['longest_gap_hours']}h")

## 1. Coverage map

Where are the gaps in the recording? Dark cells = hours with few or no epochs (charging, watch off, sync issues).

In [ ]:
coverage_heatmap(activity, timezone=TIMEZONE, title=f"Coverage — subject {SUBJECT_ID}").show()

## 2. Double-plotted actogram (full series)

Standard chronobiology view. The N24 signature is a **diagonal stripe** of activity drifting across calendar hours over weeks — the rhythm runs slower than the 24h day.

In [ ]:
double_plot_actogram(activity, timezone=TIMEZONE, bin_minutes=15, title=f"Actogram — subject {SUBJECT_ID}").show()

## 3. Average 24h profile

For an entrained subject this would peak around mid-afternoon. For an N24 subject the profile flattens because activity rotates across hours.

In [ ]:
average_24h_profile(activity, timezone=TIMEZONE, title=f"Average 24h profile — subject {SUBJECT_ID}").show()

## 4. NPCRA on sliding 14-day windows

Per `PROTOCOL.md` we compute IS, IV, RA, CFI on rolling 14-day windows (step 1 day). This shows the evolution of the rhythm over the recording, robust to local gaps.

In [ ]:
WINDOW_DAYS = 14
STEP_DAYS = 1

df = activity.copy()
df["ts_local"] = df["timestamp"].dt.tz_convert(TIMEZONE)
first_date = df["ts_local"].dt.normalize().min()
last_date = df["ts_local"].dt.normalize().max()

rows = []
current = first_date
while current + pd.Timedelta(days=WINDOW_DAYS) <= last_date:
    window_end = current + pd.Timedelta(days=WINDOW_DAYS)
    mask = (df["ts_local"] >= current) & (df["ts_local"] < window_end)
    arr = df.loc[mask, "activity"].to_numpy()
    if len(arr) < EPOCHS_PER_DAY * WINDOW_DAYS * 0.6:
        # < 60% of expected epochs in window — skip
        current += pd.Timedelta(days=STEP_DAYS)
        continue
    is_val = interdaily_stability(arr, EPOCHS_PER_DAY)
    iv_val = intradaily_variability(arr)
    res = l5_m10(arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
    ra = relative_amplitude(res.l5_value, res.m10_value)
    cfi = circadian_function_index(is_val, iv_val, ra)
    rows.append({"window_start": current, "IS": is_val, "IV": iv_val, "RA": ra, "CFI": cfi})
    current += pd.Timedelta(days=STEP_DAYS)

windows = pd.DataFrame(rows)
print(f"{len(windows)} valid 14-day windows")
print(windows[["IS", "IV", "RA", "CFI"]].describe().round(3))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=["IS", "IV", "RA", "CFI"], vertical_spacing=0.04)
colors = [TEAL, AMBER, CYAN, "#7a9aaa"]
for i, (col, color) in enumerate(zip(["IS", "IV", "RA", "CFI"], colors), start=1):
    fig.add_trace(
        go.Scatter(x=windows["window_start"], y=windows[col], mode="lines+markers",
                   line=dict(color=color, width=1.5), marker=dict(size=4), name=col),
        row=i, col=1,
    )
fig.update_layout(title=f"NPCRA evolution on 14-day sliding windows — subject {SUBJECT_ID}", height=700, showlegend=False)
fig.update_xaxes(title_text="window start", row=4, col=1)
fig.show()

## 5. Tau estimation with bootstrap 95% CI

On the full series. Bootstrap with 1000 resamples to quantify the precision of the slope of the M10-phase regression.

In [ ]:
full_arr = activity.sort_values("timestamp")["activity"].to_numpy()
point = estimate_tau(full_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
ci = bootstrap_tau_ci(full_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY, n_iter=1000, seed=42)

print(f"Full series tau estimate: {point.tau_hours:.4f} h")
print(f"R² of M10-phase regression: {point.r_squared:.3f}")
print(f"Bootstrap 95% CI: [{ci.ci_low_hours:.4f}, {ci.ci_high_hours:.4f}]")
print(f"Daily drift: {point.slope_hours_per_day*60:.2f} minutes per calendar day")
print(f"Days used: {ci.n_days_used}")

In [ ]:
m10_phase_drift_plot(activity, timezone=TIMEZONE, title=f"M10 phase drift — subject {SUBJECT_ID}").show()

## 6. Comparison with published norms

Z-scores against the healthy adult distributions (Van Someren 1999, Ortiz-Tudela 2010, Witting 1990) and the N24 range (Sack 2007, Hayakawa 2005). Negative z-scores below the healthy mean ; positive z-scores above.

In [ ]:
norms = json.loads((Path("..") / "data" / "reference" / "norms.json").read_text())
healthy = norms["healthy_adults"]
n24_pub = norms["N24_published_cases"]

personal_is = windows["IS"].median()
personal_iv = windows["IV"].median()
personal_ra = windows["RA"].median()
personal_cfi = windows["CFI"].median()

def z(value, ref):
    return (value - ref["mean"]) / ref["sd"]

comparison = pd.DataFrame([
    {"metric": "IS", "personal (median 14d windows)": round(personal_is, 3),
     "healthy mean ± sd": f"{healthy['IS']['mean']:.2f} ± {healthy['IS']['sd']:.2f}",
     "z-score vs healthy": round(z(personal_is, healthy["IS"]), 2),
     "N24 threshold (lit.)": f"< {n24_pub['IS_typical']['threshold']}"},
    {"metric": "IV", "personal (median 14d windows)": round(personal_iv, 3),
     "healthy mean ± sd": f"{healthy['IV']['mean']:.2f} ± {healthy['IV']['sd']:.2f}",
     "z-score vs healthy": round(z(personal_iv, healthy["IV"]), 2),
     "N24 threshold (lit.)": "(higher)"},
    {"metric": "RA", "personal (median 14d windows)": round(personal_ra, 3),
     "healthy mean ± sd": f"{healthy['RA']['mean']:.2f} ± {healthy['RA']['sd']:.2f}",
     "z-score vs healthy": round(z(personal_ra, healthy["RA"]), 2),
     "N24 threshold (lit.)": "(lower)"},
    {"metric": "CFI", "personal (median 14d windows)": round(personal_cfi, 3),
     "healthy mean ± sd": f"{healthy['CFI']['mean']:.2f} ± {healthy['CFI']['sd']:.2f}",
     "z-score vs healthy": round(z(personal_cfi, healthy["CFI"]), 2),
     "N24 threshold (lit.)": "(lower)"},
    {"metric": "tau (h)", "personal (median 14d windows)": round(point.tau_hours, 3),
     "healthy mean ± sd": "24.00 ± 0.10",
     "z-score vs healthy": round((point.tau_hours - 24.0) / 0.10, 2),
     "N24 threshold (lit.)": f"{n24_pub['tau_hours']['range_min']}–{n24_pub['tau_hours']['range_max']}"},
])
comparison

## Key results — to fill in after running

Replace the bullet stubs with the observed numbers and your interpretation against the preregistered hypotheses (`PROTOCOL.md` H1–H4):

- **H1 — tau > 24.0h** : observed tau = ___ h with 95% CI [___, ___]. *Confirmed* / *not confirmed*.
- **H2 — IS < 0.40** : median IS over 14-day windows = ___. *Confirmed* / *not confirmed*.
- **H3 — RA < 0.80** : median RA over 14-day windows = ___. *Confirmed* / *not confirmed*.
- **H4 — drift in 2016–2018 episodes** : not testable on the current dataset (movement coverage starts 2024-12). Re-evaluate with sleep_intervals.parquet if older sleep data is available.

Notes on coverage : ___% over ___ days, longest gap ___h. Discuss whether gaps materially affect any of the above conclusions.

*Once filled in, this section becomes the source for the corresponding paragraphs in `PROTOCOL.md` results and the manuscript draft.*